# HydroSight TN - Preprocessing

This notebook standardizes raster layers, rasterizes thematic vectors, derives lineament density, TWI, drainage density, and produces classified suitability rasters.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocess import (
    classify_layer,
    compute_twi,
    derive_drainage_density,
    extract_lineament_density,
    rasterize_vector,
    standardize_layer,
)

raw_dir = project_root / 'data' / 'raw'
std_dir = project_root / 'data' / 'standardized'
cls_dir = project_root / 'data' / 'classified'
std_dir.mkdir(parents=True, exist_ok=True)
cls_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
slope_std = standardize_layer(raw_dir / 'slope.tif', std_dir / 'slope.tif')
dem_std = standardize_layer(raw_dir / 'dem.tif', std_dir / 'dem.tif')
ndvi_std = standardize_layer(raw_dir / 'ndvi.tif', std_dir / 'ndvi.tif')
lulc_std = standardize_layer(raw_dir / 'lulc.tif', std_dir / 'lulc.tif')
rainfall_std = standardize_layer(raw_dir / 'rainfall.tif', std_dir / 'rainfall.tif')
b8_std = standardize_layer(raw_dir / 'sentinel2_B8.tif', std_dir / 'sentinel2_b8.tif')

lineament = extract_lineament_density(b8_std, std_dir / 'lineament_density.tif')
twi = compute_twi(dem_std, std_dir / 'twi.tif')
drainage = derive_drainage_density(dem_std, std_dir / 'drainage_density.tif')
lineament, twi, drainage

In [ ]:
geology_map = {'Alluvium': 1, 'Laterite': 2, 'Charnockite': 3, 'Granite_Gneiss': 4, 'Crystalline': 5}
geomorphology_map = {'Flood_Plain': 1, 'Pediplain': 2, 'Valley_Fill': 3, 'Pediment': 4, 'Denudational_Hill': 5, 'Rocky': 6}
soil_map = {'Sandy_Loam': 1, 'Alluvial': 2, 'Red_Loam': 3, 'Black_Cotton': 4, 'Sandy_Clay': 5, 'Rocky': 6}

geology_r = rasterize_vector(raw_dir / 'geology.shp', slope_std, 'geology', std_dir / 'geology.tif', value_map=geology_map)
geomorphology_r = rasterize_vector(raw_dir / 'geomorphology.shp', slope_std, 'geomorphology', std_dir / 'geomorphology.tif', value_map=geomorphology_map)
soil_r = rasterize_vector(raw_dir / 'soil.shp', slope_std, 'soil', std_dir / 'soil.tif', value_map=soil_map)

class_paths = {
    'geology': classify_layer(geology_r, 'geology', cls_dir / 'geology.tif'),
    'geomorphology': classify_layer(geomorphology_r, 'geomorphology', cls_dir / 'geomorphology.tif'),
    'soil': classify_layer(soil_r, 'soil', cls_dir / 'soil.tif'),
    'lulc': classify_layer(lulc_std, 'lulc', cls_dir / 'lulc.tif'),
    'slope': classify_layer(slope_std, 'slope', cls_dir / 'slope.tif'),
    'rainfall': classify_layer(rainfall_std, 'rainfall', cls_dir / 'rainfall.tif'),
    'lineament_density': classify_layer(lineament, 'lineament_density', cls_dir / 'lineament_density.tif'),
    'drainage_density': classify_layer(drainage, 'drainage_density', cls_dir / 'drainage_density.tif'),
    'twi': classify_layer(twi, 'twi', cls_dir / 'twi.tif'),
    'ndvi': classify_layer(ndvi_std, 'ndvi', cls_dir / 'ndvi.tif'),
}
class_paths